In [1]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pathlib import Path
from typing import Literal

from pydantic import BaseModel


class RepositoryFinding(BaseModel):
    file_path: str
    title: str
    severity: Literal["low", "medium", "high"]
    evidence: str
    explanation: str
    recommendation: str

In [2]:


PROJECT_ROOT = Path.cwd()
REPOSITORY_ROOT = PROJECT_ROOT / "sample_repository"

print(REPOSITORY_ROOT)
print(REPOSITORY_ROOT.exists())

/Users/theo/Documents/Projects/programingProjects/agentic_AI_project/agentic-ai/sample_repository
True


In [3]:
model = OpenAIChatModel(
    model_name="qwen3:4b-instruct",
    provider=OllamaProvider(
        base_url="http://localhost:11434/v1"
    ),
)
# agent = Agent(model)
agent = Agent(
    model,
    output_type=RepositoryFinding,
)

```py
agent = Agent(
    model,
    output_type=RepositoryFinding,
)
```
tells the agent:
> the final answer must be a `RepositoryFinding`  type.

Because you created list_repository_files() and read_repository_file() using:
@agent.tool_plain
those tools belong to the old agent object.
When you replace the agent, you need to register the tools again.


---

## creating a filesystem tool

In [4]:
@agent.tool_plain
def list_repository_files() -> list[str]:
    """Return the relative paths of files in the repository."""
    return [
        str(path.relative_to(REPOSITORY_ROOT))
        for path in REPOSITORY_ROOT.rglob("*")
        if path.is_file()
    ]
    
    

@agent.tool_plain
def read_repository_file(path: str) -> str:
    """Read a text file from the repository."""
    file_path = (REPOSITORY_ROOT / path).resolve()

    if not file_path.is_relative_to(REPOSITORY_ROOT.resolve()):
        raise ValueError("Path is outside the repository.")

    if not file_path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    return file_path.read_text(encoding="utf-8")

In [5]:
files = list_repository_files()
print(files)
print(read_repository_file("user_service.py"))

['test_user_service.py', 'app.py', 'user_service.py']
USERS = {
    "1": "Theo",
    "2": "Alice",
    "3": "Bob",
}


def get_user_name(user_id: str) -> str:
    return USERS[user_id]


---

---

---

## making the agent use do something

In [ ]:
result = await agent.run(
    """
    Inspect the repository.

    First determine which files exist.
    Then inspect the files relevant to the user service.

    Explain:
    1. Which files are relevant.
    2. What the user service does.
    3. One potential problem in the implementation.

    You must use the available repository tools.
    """
)

print(result.output)

<class '__main__.RepositoryFinding'>


In [13]:
def print_repository_finding(finding: RepositoryFinding) -> None:
    print(f"File:     {finding.file_path}")
    print(f"Title:    {finding.title}")
    print(f"Severity: {finding.severity}")
    print()
    print("Evidence:")
    print(finding.evidence)
    print()
    print("Explanation:")
    print(finding.explanation)
    print()
    print("Recommendation:")
    print(finding.recommendation)

In [12]:
print_repository_finding(result.output)

File: user_service.py
Title: Input Validation Missing in User Service
Severity: high

Evidence:
The get_user_name function does not validate if the user_id exists in the USERS dictionary. If an invalid user_id is provided, a KeyError will be raised.

Explanation:
The function get_user_name(user_id: str) retrieves a user name from the USERS dictionary without checking whether the user_id is present. This causes a KeyError when an invalid user_id is provided, leading to runtime crashes.

Recommendation:
Modify the get_user_name function to first check if the user_id exists in the USERS dictionary. If not, return a default value (e.g., 'User not found') or raise a custom exception with a clear error message. Add input validation in app.py to ensure only valid user_ids are passed to get_user_name.


There are now two different levels of execution.
Your Python program
Your notebook controls:
```
Agent
Tools
Ollama connection
Repository access
```

The model
The model decides:
```
Which tool should I use?
What arguments should I provide?
Do I need another tool?
When should I stop?
```
That distinction is the central idea we have been building toward.